# Competitive predictive refresh

This benchmark measures the current autospec pipeline on all thirteen
predictive panels of the frozen competitive shootout. It preserves the
original comparator measurements and compares each current task with the
strongest complete recorded non-mboostrs method across both search tiers.
Every failed run remains in the denominator.

In [ ]:
import json
import sys
from pathlib import Path

import polars as pl
from IPython.display import Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Cargo.toml").exists())
sys.path.insert(0, str(ROOT))
from benches.report_competitive_refresh import DIRECTORY, headline
from docs.book.cases.bench_tables import table

summary = json.loads((DIRECTORY / "small_summary.json").read_text())
card = json.loads((DIRECTORY / "small_scorecard.json").read_text())
display(headline(summary))

## Study design

The [frozen protocol](competitive_refresh/PROTOCOL.md) defines ten real-data
panels with three outer splits each and three generated panels with five
independent draws each: 45 jobs per search tier. This page reports the current
small tier, with autospec budget 32. It does not infer a current tuned-tier
result from the small tier. Historical comparators retain both tiers.

Autospec receives the data and response semantics, with Gaussian regression
or binomial classification as appropriate, and no formula or planted truth.
Its initial stopping grid is 200 updates, with step size 0.1. The supplied
training/validation fold chooses the specification and stopping iteration.
Separate calibration rows determine predictive calibration; test rows enter
the evaluator after those choices. The original runner supplies all data,
preprocessing, splits, settings and measurements.

The real panels include independent-row, future-time and new-region outer
assessments. The shared supplied validation design remains the original
study's single holdout; this refresh is a paired development regression, not
a claim that inner validation matches every deployment design. The three
synthetic panels have known additive, interaction and correlated-proxy truth.

Jobs run serially in fresh processes with one thread and a 240-second limit.
Each record retains the source revision, loaded package and binary hashes,
script hashes and job identity. Input and partition digests must match the
historical rows before a comparison can be reported. Historical runtimes
cannot establish a current implementation speed ratio.

## Task-level gaps

The target is at most 5% excess test loss on every task. Regression loss is
mean squared error (MSE); binary loss is log loss. Both are smaller for better
predictions. The reference is the complete historical method/tier with lowest
mean loss on the same task. Selecting that reference using test outcomes gives
a retrospective oracle comparison, rather than a prospective model choice.
The current tier is fixed across tasks; its test errors never select a tier.

A task is complete only when every planned outer draw succeeds. Incomplete
tasks have no relative-loss verdict, even when their successful subset looks
competitive. Standard deviations describe spread over overlapping real-data
splits or independent synthetic draws. They are not population confidence
intervals.

In [ ]:
gaps = pl.DataFrame(summary["gaps"])
display(table(
    gaps.select("task", "completed", "planned", "failed", "current_loss", "loss_sd",
                "best_method", "best_tier", "best_loss", "excess_pct", "within_5pct"),
    stub="task", stubhead="Task",
    title="Current small-tier autospec: every task and every failure",
    subtitle="Test MSE or binary log loss; excess in percent, lower is better",
    spanners={"Accounting": ["completed", "planned", "failed"],
              "Current prediction": ["current_loss", "loss_sd"],
              "Historical oracle": ["best_method", "best_tier", "best_loss"],
              "Target": ["excess_pct", "within_5pct"]},
    formats={"excess_pct": 1},
    notes=("All outer draws must complete before a task passes. Missing relative losses remain unresolved.",
           "The best method/tier is chosen retrospectively from complete non-mboostrs historical pipelines; tuning budgets differ."),
))

In [ ]:
display(Markdown(
    f"Of {summary['planned_tasks']} measured tasks, **{summary['passing_tasks']}** meet the "
    f"5% target. The run records **{len(summary['failures'])} failed jobs** out of "
    f"{summary['planned_jobs']} planned jobs. This is the pre-repair baseline at "
    f"source `{summary['assessment']['source_revision'][:12]}`."
))

## Scorecard profile

The assessment uses `mboostrs-scorecard/2.1`, with the exact content hash
recorded before fitting. Task-level predictive validity (D11) scores 3 when
all draws complete and mean outer loss improves on the prespecified same-tier
Ridge/logistic reference. It scores 1 when the current pipeline fails to beat
that simple reference. A failed task remains unmeasured. This reference score
and the stronger 5% competitor gate answer different questions.

Auditability (D10) is graded for the instrument's retained code, environment,
loaded binary, input, partition and output identities and its reproducible
assessment. The other dimensions are unmeasured by this experiment; the
profile has no aggregate. In particular, recovering a predictive split does
not establish relationship recovery, explanation fidelity or causal validity.
Historical comparators, unequal search budgets and known tasks cannot support
D11 level 4 or 5.

In [ ]:
profiles = pl.DataFrame(card["task_profiles"])
display(table(
    profiles.select("task", "status", "score", "justification"),
    stub="task", stubhead="Task",
    title="Predictive validity (D11): task-level anchor assessments",
    subtitle="Scores 1–5, higher is better; failed tasks are unmeasured",
    spanners={"Assessment": ["status", "score", "justification"]},
    notes=("Level 3 means useful outer performance over the prespecified simple reference; it does not imply the 5% competitor target is met.",),
))

In [ ]:
profile = pl.DataFrame([
    {"dimension": f"{d['id']} {d['name']}", "status": d["status"], "score": d["score"],
     "justification": d["justification"] or "Outside this instrument's measurements; D11 is reported per task."}
    for d in card["dimensions"]
])
display(table(
    profile, stub="dimension", stubhead="Dimension",
    title="Knowledge-discovery scorecard 2.1: instrument coverage",
    subtitle="Scores 1–5, higher is better; unmeasured dimensions have no numeric score",
    spanners={"Assessment": ["status", "score", "justification"]},
    notes=("No weighted aggregate is defined for an incomplete profile. Task-level D11 grades are not averaged.",),
))

## Joint-replacement diagnostic

An exploratory counterfactual measures whether the remaining gap comes from
available capacity that the search did not reach. On protein, bike_iid and
diamonds, fit one joint tree over all predictors. Choose its depth from 3 and
6 using the original supplied validation fold; each path has at most 1,200
updates with step size 0.05. Score test rows only after choosing the depth and
stopping point. The table below uses all three outer splits on each task.

The current final joint-replacement comparison (a joint tree tested against the searched formula) is restricted to geographic
column names and numeric inputs; residual-tree additions (a tree fitted to the residuals of the searched formula) and pair-specific
candidates (candidate forms for one pair of predictors) are different specifications. The counterfactual identifies a
search opportunity, rather than validating a new default policy. It nearly
closes or reverses these task gaps, while protein and bike_iid still exceed
5%. Some paths stop near the cap, so convergence remains unverified. These
known development splits provide no independent confirmation.


In [ ]:
probe = json.loads((DIRECTORY / "joint_replacement_probe.json").read_text())
probe_frame = pl.DataFrame([
    {"task": r["task"], "seed": r["seed"], "loss": r["loss"]}
    for r in probe["results"]
]).group_by("task").agg(pl.col("loss").mean().alias("joint_only_loss"))
comparison = gaps.select("task", "current_loss", "best_method", "best_tier", "best_loss").join(probe_frame, on="task")
comparison = comparison.with_columns(
    (100 * (pl.col("joint_only_loss") / pl.col("best_loss") - 1)).alias("joint_excess_pct")
)
display(table(
    comparison, stub="task", stubhead="Task",
    title="Available joint capacity on three completed development panels",
    subtitle="Mean test MSE and excess in percent; lower is better",
    spanners={"Current": ["current_loss"], "Counterfactual": ["joint_only_loss", "joint_excess_pct"],
              "Historical oracle": ["best_method", "best_tier", "best_loss"]},
    formats={"joint_excess_pct": 1},
    notes=("Three paired outer splits; depth and positive stopping iteration chosen by validation risk, never by test error.",
           "Two-candidate diagnostic with different search scope and budget; it is not a promoted autospec policy or a new independent confirmation."),
))

## Failures and follow-up

The refresh retains the six observed failures on the housing tasks. They come from replacing a single validation holdout with an invalid grouped spatial validation scheme. A run with a corrected scheme writes new records and does not overwrite this baseline. The pair-form replay correction is measured separately in [](pair_form_stopping.ipynb). A candidate-cap coupling remains open.

The binary cancer task is unstable. An exploratory all-linear specification improves mean outer log loss, while its validation minima are worse on all three draws. Offering that all-linear specification to the search and choosing it by its validation loss would therefore not resolve the binary cancer failure. Its mechanism requires a separate whole-procedure investigation.

A residual prediction gap can come from missing representation, a search that
never offers available capacity, an insufficient stopping path or deployment
shift. These require different counterfactual comparisons. The current table
identifies where to investigate; it cannot assign a mechanism from test loss
alone. Further fixes and their confirmation runs are recorded separately.

The two models for the first draw (draw 0) of the cancer panel behind this task's gap, the search's seven-term
model and the tuned linear model holding the best mean log loss in the study's battery of tasks (0.0901), are read term by term in [the draw-0 interpretation
record](cancer_battery_draw0_interpretation.md). Effective parameters count the flexibility a penalized fit uses, as the trace of its hat matrix (the matrix that maps the response to the fitted values). Headline results: the linear
model carries 27.08 effective parameters of its 31 nominal (thirty weights and
an intercept; ridge hat trace at the winning penalty, 1.0), while the search's
model carries 18.49 effective parameters (conditional hat trace) from 225
basis columns. Its draw-0 interaction surface (mean concave points × worst
texture) claims the two measurements amplify each other, and its fitted rows
show that pattern in the middle of the range; the same pair's surface in draw
2 claims the opposite (substitution), against rows showing the draw-0 pattern.
Refits of that record with the code in this repository reproduced the recorded
scores with a gap of 0.0 on every recorded score: the three draws' search
models on validation, calibration and test, and the draw-0 linear model on its
recorded test log loss.

## Artifacts and reproduction

The [raw current jobs](competitive_refresh/small.json),
[assessment](competitive_refresh/small_summary.json),
[scorecard](competitive_refresh/small_scorecard.json),
[headline table](competitive_refresh/small_headline.html) and
[complete per-task comparison](competitive_refresh/small_per_task.html)
preserve the evidence and failure accounting. Per-draw paired losses are in
the assessment. The complete comparison includes every historical method and
both search tiers, with completion counts alongside successful-fit means.

```bash
.venv/bin/python -m benches.refresh_competitive --tier small
.venv/bin/python -m benches.report_competitive_refresh --tier small
```

The original shootout and its reference hash remain unchanged. A run with a different library creates a new source identity, so keep the baseline artifact before any new run. The broader causal,
distributional, interval and analyst-workflow targets remain outside these
predictive panels.